# Building the Frontend for a Full-Stack LangGraph App

## Step-by-Step Guide: A Chat Interface for the Customer Service Agent

In the previous notebook (`065-LG1-top51-backend.ipynb`), we created a **FastAPI backend** that wraps the LangGraph customer service agent from notebook 028. The backend exposes a `POST /chat` endpoint that receives a message and returns the agent's response.

Now we need the other half: a **frontend** — a web page where customers can type messages and see the agent's responses. This is what the customer actually sees and interacts with.

We'll build it using **Next.js** (React), the same framework used in the To Do app frontend (`051-todoapp-LC-frontend`).

**In this notebook, we will create the frontend step by step.**

## What is a Frontend? (Quick Refresher)

The frontend is the **website** that the customer sees in their browser. It has two jobs:

1. **Display** a chat window where the customer can type messages.
2. **Communicate** with the backend by sending HTTP requests and showing the responses.

```
┌──────────────────────────────────────────────────┐
│                    Browser                       │
│                                                  │
│  ┌─────────────────────────────────────────────┐ │
│  │  BigCompany Inc. Customer Support           │ │
│  │                                             │ │
│  │  ┌──────────────────────────────────┐       │ │
│  │  │Agent: Hello! How can I help you? │       │ │
│  │  └──────────────────────────────────┘       │ │
│  │       ┌───────────────────────────────────┐ │ │
│  │       │ You: What's the status of ORD-1002│ │ │
│  │       └───────────────────────────────────┘ │ │
│  │  ┌──────────────────────────────────┐       │ │
│  │  │ Agent: Your order is shipped...  │       │ │
│  │  └──────────────────────────────────┘       │ │
│  │                                             │ │
│  │  ┌──────────────────────────────┬──────┐    │ │
│  │  │ Type your message...         │ Send │    │ │
│  │  └──────────────────────────────┴──────┘    │ │
│  └─────────────────────────────────────────────┘ │
└──────────────────────────────────────────────────┘
```

The frontend **does not** contain any AI logic. It simply sends the customer's message to the backend, waits for the response, and displays it.

## How Does This Compare to the To Do App Frontend?

The To Do app frontend (`051-todoapp-LC-frontend`) shows a list of tasks. Users can add, edit, delete, and filter tasks. Each action sends an HTTP request to the backend.

Our chat frontend shows a conversation. Users type messages and receive responses. Each message sends **one HTTP request** to the backend (`POST /chat`).

| Feature | To Do App Frontend (051) | Our Chat Frontend |
|---------|-------------------------|-------------------|
| Framework | Next.js (React) | Next.js (React) |
| Main component | `todo-list.js` (list of tasks) | `chat.js` (chat window) |
| User action | Add/edit/delete tasks | Send messages |
| API calls | Multiple (POST, GET, PUT, DELETE) | One (POST /chat) |
| React hooks | `useState`, `useEffect`, `useCallback`, `useRef` | `useState`, `useRef`, `useEffect` |
| Styling | CSS Modules | CSS Modules |

Same tools, same patterns, different purpose.

## Project Structure

Here are the files we will create:

```
frontend/
├── .env                      # Backend API URL
├── package.json              # Dependencies (next, react)
├── next.config.mjs           # Next.js configuration
├── jsconfig.json             # JavaScript path aliases
├── pages/
│   ├── _app.js               # App wrapper (loads global CSS)
│   ├── _document.js          # HTML document structure
│   └── index.js              # Main page
├── components/
│   ├── layout.js             # Page layout (title, centering)
│   └── chat.js               # The chat component (the star of the show!)
└── styles/
    ├── globals.css            # Global styles (body, fonts)
    ├── layout.module.css      # Layout styles
    └── chat.module.css        # Chat component styles
```

Compare with the To Do app frontend (`051-todoapp-LC-frontend/047-todoapp-frontend`):

```
047-todoapp-frontend/
├── .env                      # Backend API URL           ← same role
├── package.json              # Dependencies              ← same role
├── next.config.mjs           # Next.js configuration     ← same role
├── jsconfig.json             # Path aliases              ← same role
├── pages/
│   ├── _app.js               # App wrapper               ← same role
│   ├── _document.js          # HTML document             ← same role
│   └── index.js              # Main page                 ← same role
├── components/
│   ├── layout.js             # Layout                    ← same role
│   ├── todo-list.js          # To Do list component      ← replaced by chat.js
│   └── todo.js               # Single To Do item         ← not needed (we don't have items)
└── styles/
    ├── globals.css            # Global styles             ← same role
    ├── layout.module.css      # Layout styles             ← same role
    ├── todo-list.module.css   # To Do list styles         ← replaced by chat.module.css
    └── todo.module.css        # To Do item styles         ← not needed
```

The structure is almost identical. The main difference is that `todo-list.js` and `todo.js` are replaced by a single `chat.js`.

Let's build it file by file!

## Step 1: Create the Project Folders

We need three folders: `pages/`, `components/`, and `styles/`.

In [ ]:
import os

# Create the frontend folders
os.makedirs("frontend/pages", exist_ok=True)
os.makedirs("frontend/components", exist_ok=True)
os.makedirs("frontend/styles", exist_ok=True)
os.makedirs("frontend/public", exist_ok=True)

print("Folders created!")
print("  frontend/")
print("  frontend/pages/")
print("  frontend/components/")
print("  frontend/styles/")
print("  frontend/public/")

## Step 2: Create the `.env` File (Backend API URL)

The frontend needs to know **where the backend is running** so it can send messages to it.

In Next.js, environment variables that start with `NEXT_PUBLIC_` are available in the browser. We use `NEXT_PUBLIC_API_URL` to store the backend URL.

This is exactly the same as the To Do app's `.env` file.

In [ ]:
%%writefile frontend/.env
NEXT_PUBLIC_API_URL=http://localhost:8000

### Compare with the To Do app `.env`

To Do app: `NEXT_PUBLIC_API_URL=http://localhost:8000`

Our chat app: `NEXT_PUBLIC_API_URL=http://localhost:8000`

Identical! Both frontends talk to a backend on port 8000.

## Step 3: Create `package.json` (Dependencies)

This file lists the JavaScript packages our frontend needs and defines the commands to run it.

The To Do app needed `lodash` (for debouncing input changes). Our chat app doesn't need any extra libraries — just **Next.js** and **React**.

| Package | What It Does |
|---------|--------------|
| `next` | The React framework (handles routing, server-side rendering, etc.) |
| `react` | The UI library (components, hooks, state management) |
| `react-dom` | Renders React components in the browser |

In [ ]:
%%writefile frontend/package.json
{
  "name": "bigcompany-chat-frontend",
  "version": "0.1.0",
  "private": true,
  "scripts": {
    "dev": "next dev",
    "build": "next build",
    "start": "next start"
  },
  "dependencies": {
    "next": "16.1.4",
    "react": "19.2.3",
    "react-dom": "19.2.3"
  }
}

### The scripts explained

| Command | What It Does |
|---------|--------------|
| `npm run dev` | Starts the development server (with hot reload) |
| `npm run build` | Builds the app for production |
| `npm start` | Runs the production build |

During development, we'll use `npm run dev`. It starts on **http://localhost:3000**.

## Step 4: Create the Configuration Files

These are small configuration files that Next.js needs. They are the same as the To Do app's config files.

In [ ]:
%%writefile frontend/next.config.mjs
/** @type {import('next').NextConfig} */
const nextConfig = {
  reactStrictMode: true,
};

export default nextConfig;

In [ ]:
%%writefile frontend/jsconfig.json
{
  "compilerOptions": {
    "paths": {
      "@/*": ["./*"]
    }
  }
}

- **`next.config.mjs`**: Enables React Strict Mode, which helps catch common mistakes during development.
- **`jsconfig.json`**: Sets up the `@/` path alias, so we can write `import styles from '@/styles/globals.css'` instead of `import styles from '../../styles/globals.css'`.

These are identical to the To Do app's config files.

## Step 5: Create the Page Structure

In Next.js, the `pages/` folder defines the routes of the website:
- `pages/index.js` → the home page (`/`)
- `pages/_app.js` → wraps every page (loads global styles)
- `pages/_document.js` → the HTML document structure

These files are the **same boilerplate** in every Next.js app. The To Do app has the exact same files.

In [ ]:
%%writefile frontend/pages/_app.js
import "@/styles/globals.css";

export default function App({ Component, pageProps }) {
  return <Component {...pageProps} />;
}

In [ ]:
%%writefile frontend/pages/_document.js
import { Html, Head, Main, NextScript } from "next/document";

export default function Document() {
  return (
    <Html lang="en">
      <Head />
      <body>
        <Main />
        <NextScript />
      </body>
    </Html>
  );
}

- **`_app.js`**: Loads the global CSS file so styles are available on every page. This is identical to the To Do app.
- **`_document.js`**: Defines the HTML document structure (`<html>`, `<head>`, `<body>`). This is identical to the To Do app.

## Step 6: Create the Styles

We need three CSS files:

1. **`globals.css`** — Base styles for the entire page (font, background).
2. **`layout.module.css`** — Styles for the layout wrapper (centering, title).
3. **`chat.module.css`** — Styles for the chat component (messages, input, bubbles).

We use **CSS Modules** (files ending in `.module.css`). This is the same approach the To Do app uses. CSS Modules keep styles scoped to their component, so they don't accidentally affect other parts of the page.

In [ ]:
%%writefile frontend/styles/globals.css
* {
  box-sizing: border-box;
  margin: 0;
  padding: 0;
}

body {
  font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
  background-color: #fafafa;
  display: flex;
  justify-content: center;
  padding-top: 30px;
}

In [ ]:
%%writefile frontend/styles/layout.module.css
.layout {
  width: 500px;
}

.title {
  text-align: center;
  font-size: 24px;
  margin-bottom: 15px;
  color: #333;
}

Now the most important CSS file — the styles for the chat component. This creates the look of the chat window: message bubbles, the input area, and a loading indicator.

Compare with the To Do app:
- To Do app had `todo-list.module.css` (container, input, filter buttons) and `todo.module.css` (todo row, checkbox, delete button).
- Our app has `chat.module.css` (container, messages area, input form, message bubbles).

In [ ]:
%%writefile frontend/styles/chat.module.css
/* ============================================================
   Chat Component Styles
   ============================================================ */

/* The main chat container — a tall box with messages on top and input at the bottom */
.container {
  width: 500px;
  height: 600px;
  border: 1px solid #ddd;
  border-radius: 8px;
  background-color: white;
  display: flex;
  flex-direction: column;
}

/* The scrollable area where messages appear */
.messages {
  flex: 1;
  overflow-y: auto;
  padding: 20px;
}

/* Shown when there are no messages yet */
.emptyState {
  text-align: center;
  color: #999;
  margin-top: 200px;
  padding: 0 40px;
  line-height: 1.6;
}

/* Each message bubble */
.message {
  margin-bottom: 15px;
  padding: 10px 15px;
  border-radius: 8px;
  max-width: 85%;
}

/* Customer messages: blue background, aligned to the right */
.user {
  background-color: #e3f2fd;
  margin-left: auto;
}

/* Agent messages: gray background, aligned to the left */
.agent {
  background-color: #f5f5f5;
  margin-right: auto;
}

/* The label above each message ("You" or "Agent") */
.sender {
  font-size: 11px;
  font-weight: bold;
  margin-bottom: 4px;
  color: #888;
}

/* The message text */
.content {
  line-height: 1.5;
  white-space: pre-wrap;
  font-size: 14px;
}

/* The input form at the bottom */
.inputForm {
  display: flex;
  padding: 15px;
  border-top: 1px solid #ddd;
  gap: 10px;
}

/* The text input */
.input {
  flex: 1;
  padding: 10px;
  border: 1px solid #ddd;
  border-radius: 4px;
  font-size: 14px;
  outline: none;
}

.input:focus {
  border-color: #4a90d9;
}

/* The send button */
.sendBtn {
  padding: 10px 20px;
  background-color: #4a90d9;
  color: white;
  border: none;
  border-radius: 4px;
  cursor: pointer;
  font-size: 14px;
}

.sendBtn:hover {
  background-color: #357abd;
}

/* Disabled state when waiting for a response */
.sendBtn:disabled {
  background-color: #ccc;
  cursor: not-allowed;
}

## Step 7: Create `components/layout.js` (The Layout)

The layout component wraps the page content and adds a title. This is the same pattern as the To Do app's `layout.js`.

In [ ]:
%%writefile frontend/components/layout.js
import styles from '../styles/layout.module.css'

export default function Layout(props) {
  return (
    <div className={styles.layout}>
      <h1 className={styles.title}>BigCompany Inc. Support</h1>
      {props.children}
    </div>
  )
}

### Compare with the To Do app layout

To Do app:
```jsx
export default function Layout(props) {
  return (
    <div className={styles.layout}>
      <h1 className={styles.title}>To Do</h1>
      {props.children}
    </div>
  )
}
```

Our chat app:
```jsx
export default function Layout(props) {
  return (
    <div className={styles.layout}>
      <h1 className={styles.title}>BigCompany Inc. Support</h1>
      {props.children}
    </div>
  )
}
```

The only difference is the title text. Everything else is the same.

## Step 8: Create `components/chat.js` (The Star of the Show!)

This is the **main component** of our frontend — the chat window. It replaces the To Do app's `todo-list.js`.

Here's what this component does:

1. **Keeps a list of messages** in state (using `useState`).
2. **Shows the messages** in a scrollable area (customer messages on the right, agent messages on the left).
3. **Has a text input** and a Send button at the bottom.
4. **When the user sends a message**: adds the message to the list, sends it to the backend via `POST /chat`, waits for the response, and adds the agent's reply to the list.
5. **Shows "Thinking..."** while waiting for the agent to respond.
6. **Auto-scrolls** to the bottom when new messages arrive.
7. **Generates a unique thread ID** for each page load, so each visit starts a fresh conversation.

### The key React concepts used

| Concept | What It Does | Also Used In To Do App? |
|---------|-------------|------------------------|
| `useState` | Stores the list of messages, the input text, and the loading state | Yes (stores todos, input, filter) |
| `useRef` | Creates a reference to the bottom of the messages area (for auto-scroll) | Yes (didFetchRef) |
| `useEffect` | Auto-scrolls to the bottom when messages change | Yes (fetches todos on mount) |
| `fetch()` | Sends HTTP requests to the backend | Yes (all CRUD operations) |
| `async/await` | Handles the asynchronous API call | Yes (fetchTodos, addToDo, etc.) |

In [ ]:
%%writefile frontend/components/chat.js
// ============================================================
// chat.js — The Chat Component
//
// This is the main component of our frontend.
// It shows a chat window where customers can talk to the
// BigCompany Inc. customer service agent.
// ============================================================

import { useState, useRef, useEffect } from 'react'
import styles from '../styles/chat.module.css'

export default function Chat() {
  // ---- STATE ----
  // messages: the list of all messages in the conversation
  //   Each message is an object: { role: 'user' or 'agent', content: 'the text' }
  const [messages, setMessages] = useState([])

  // input: what the user is currently typing in the text field
  const [input, setInput] = useState('')

  // isLoading: true while waiting for the agent's response
  const [isLoading, setIsLoading] = useState(false)

  // threadId: a unique ID for this conversation
  //   Generated once when the component loads.
  //   Same threadId = same conversation (the agent remembers previous messages).
  const [threadId] = useState(() => 'thread-' + Date.now())

  // messagesEndRef: a reference to the bottom of the messages area
  //   Used to auto-scroll to the latest message.
  const messagesEndRef = useRef(null)

  // ---- AUTO-SCROLL ----
  // Whenever the messages list changes, scroll to the bottom.
  useEffect(() => {
    messagesEndRef.current?.scrollIntoView({ behavior: 'smooth' })
  }, [messages, isLoading])

  // ---- SEND MESSAGE ----
  // This function is called when the user clicks Send or presses Enter.
  async function sendMessage(e) {
    e.preventDefault()

    // Don't send empty messages or send while already loading
    if (!input.trim() || isLoading) return

    const userMessage = input.trim()
    setInput('')  // Clear the input field

    // Step 1: Add the user's message to the list immediately
    //   (so the user sees their message right away)
    setMessages(prev => [...prev, { role: 'user', content: userMessage }])

    // Step 2: Send the message to the backend
    setIsLoading(true)

    try {
      const res = await fetch(process.env.NEXT_PUBLIC_API_URL + '/chat', {
        method: 'POST',
        headers: { 'Content-Type': 'application/json' },
        body: JSON.stringify({
          message: userMessage,
          thread_id: threadId,
        }),
      })

      if (res.ok) {
        // Step 3: Add the agent's response to the list
        const data = await res.json()
        setMessages(prev => [...prev, { role: 'agent', content: data.response }])
      } else {
        // Something went wrong on the server
        setMessages(prev => [...prev, {
          role: 'agent',
          content: 'Sorry, something went wrong. Please try again.'
        }])
      }
    } catch (error) {
      // Could not connect to the backend
      setMessages(prev => [...prev, {
        role: 'agent',
        content: 'Could not connect to the server. Is the backend running on ' +
          process.env.NEXT_PUBLIC_API_URL + '?'
      }])
    }

    setIsLoading(false)
  }

  // ---- RENDER ----
  return (
    <div className={styles.container}>

      {/* Messages area (scrollable) */}
      <div className={styles.messages}>

        {/* Empty state: shown when there are no messages yet */}
        {messages.length === 0 && (
          <div className={styles.emptyState}>
            Welcome to BigCompany Inc. support!<br />
            Send a message to start chatting with our AI agent.
          </div>
        )}

        {/* Render each message */}
        {messages.map((msg, index) => (
          <div
            key={index}
            className={`${styles.message} ${msg.role === 'user' ? styles.user : styles.agent}`}
          >
            <div className={styles.sender}>
              {msg.role === 'user' ? 'You' : 'Agent'}
            </div>
            <div className={styles.content}>{msg.content}</div>
          </div>
        ))}

        {/* Loading indicator: shown while waiting for the agent */}
        {isLoading && (
          <div className={`${styles.message} ${styles.agent}`}>
            <div className={styles.sender}>Agent</div>
            <div className={styles.content}>Thinking...</div>
          </div>
        )}

        {/* Invisible element at the bottom — used for auto-scroll */}
        <div ref={messagesEndRef} />
      </div>

      {/* Input form at the bottom */}
      <form className={styles.inputForm} onSubmit={sendMessage}>
        <input
          className={styles.input}
          type="text"
          placeholder="Type your message..."
          value={input}
          onChange={(e) => setInput(e.target.value)}
          disabled={isLoading}
        />
        <button
          className={styles.sendBtn}
          type="submit"
          disabled={isLoading || !input.trim()}
        >
          Send
        </button>
      </form>
    </div>
  )
}

### Understanding `chat.js` — The Key Parts

Let's break down the most important parts of this component:

#### 1. State Management
```jsx
const [messages, setMessages] = useState([])    // List of messages
const [input, setInput] = useState('')           // Current input text
const [isLoading, setIsLoading] = useState(false) // Is the agent thinking?
const [threadId] = useState(() => 'thread-' + Date.now())  // Unique conversation ID
```

The To Do app had similar state: `const [todos, setTodos] = useState(null)` and `const [mainInput, setMainInput] = useState('')`.

#### 2. Sending a Message to the Backend
```jsx
const res = await fetch(process.env.NEXT_PUBLIC_API_URL + '/chat', {
  method: 'POST',
  headers: { 'Content-Type': 'application/json' },
  body: JSON.stringify({ message: userMessage, thread_id: threadId }),
})
const data = await res.json()  // { response: "...", thread_id: "..." }
```

This is the same pattern as the To Do app's `addToDo()` function:
```jsx
const res = await fetch(process.env.NEXT_PUBLIC_API_URL + '/todos/', {
  method: 'POST',
  body: JSON.stringify({ name: name, completed: false }),
  headers: { 'Content-Type': 'application/json' }
})
```

Same `fetch()` pattern, different endpoint and data.

#### 3. Auto-Scrolling
```jsx
const messagesEndRef = useRef(null)

useEffect(() => {
  messagesEndRef.current?.scrollIntoView({ behavior: 'smooth' })
}, [messages, isLoading])
```

We place an invisible `<div ref={messagesEndRef} />` at the bottom of the messages. Every time the messages list changes, `useEffect` scrolls that invisible div into view. This ensures the latest message is always visible.

## Step 9: Create `pages/index.js` (The Main Page)

This is the main page of our website. It assembles the layout and the chat component together.

This follows the exact same pattern as the To Do app's `index.js`.

In [ ]:
%%writefile frontend/pages/index.js
import Head from 'next/head'
import Layout from '../components/layout'
import Chat from '../components/chat'

export default function Home() {
  return (
    <div>
      <Head>
        <title>BigCompany Inc. Customer Support</title>
        <meta name="description" content="BigCompany Inc. AI Customer Support" />
        <link rel="icon" href="/favicon.ico" />
      </Head>
      <Layout>
        <Chat />
      </Layout>
    </div>
  )
}

### Compare with the To Do app `index.js`

To Do app:
```jsx
import Layout from '../components/layout';
import ToDoList from '../components/todo-list';

export default function Home() {
  return (
    <div>
      <Head>
        <title>Full Stack Book To Do</title>
      </Head>
      <Layout>
        <ToDoList />
      </Layout>
    </div>
  )
}
```

Our chat app:
```jsx
import Layout from '../components/layout'
import Chat from '../components/chat'

export default function Home() {
  return (
    <div>
      <Head>
        <title>BigCompany Inc. Customer Support</title>
      </Head>
      <Layout>
        <Chat />
      </Layout>
    </div>
  )
}
```

The only differences: the title and `<ToDoList />` is replaced by `<Chat />`.

## All Files Created! Let's Review

Let's verify that all our frontend files were created correctly.

In [ ]:
import os

print("Frontend project structure:")
print("=" * 40)
for root, dirs, files in os.walk("frontend"):
    # Skip hidden folders and node_modules
    dirs[:] = [d for d in dirs if not d.startswith(".") and d != "node_modules" and d != ".next"]
    level = root.replace("frontend", "").count(os.sep)
    indent = "  " * level
    folder_name = os.path.basename(root)
    print(f"{indent}{folder_name}/")
    sub_indent = "  " * (level + 1)
    for file in sorted(files):
        if not file.startswith(".") or file in [".env"]:
            filepath = os.path.join(root, file)
            size = os.path.getsize(filepath)
            print(f"{sub_indent}{file} ({size} bytes)")

## Step 10: Install Dependencies and Run the Frontend

### Step 10a: Install the JavaScript packages

Open a terminal and run:

```bash
cd frontend
npm install
```

This reads `package.json` and downloads all the packages (Next.js, React, etc.) into a `node_modules/` folder. It might take a minute.

### Step 10b: Start the backend (if not already running)

The frontend needs the backend to be running so it can send messages. In a **separate terminal**:

```bash
cd backend
uvicorn main:app --reload
```

The backend starts on **http://localhost:8000**.

### Step 10c: Start the frontend

In another terminal:

```bash
cd frontend
npm run dev
```

The frontend starts on **http://localhost:3000**.

### Step 10d: Open the chat!

Open your browser and go to:

**http://localhost:3000**

You should see the chat window. Try these messages:

1. `Hi! I need help with my order.`
2. `Can you check the status of order ORD-1002?`
3. `When will it arrive?` (tests conversation memory!)
4. `What is your return policy?`

## How Everything Works Together

Here's the complete flow when a customer types a message:

```
1. Customer types "Check order ORD-1002" and clicks Send
        │
        ▼
2. chat.js adds the message to the screen immediately
   and shows "Thinking..."
        │
        ▼
3. chat.js sends POST http://localhost:8000/chat
   with {message: "Check order ORD-1002", thread_id: "thread-1707..."}
        │
        ▼
4. FastAPI backend (main.py) receives the request
   └── Routes it to routers/chat.py
        │
        ▼
5. routers/chat.py calls graph.invoke() from agent.py
        │
        ▼
6. LangGraph agent (agent.py):
   ├── Assistant node: LLM reads the message
   ├── LLM decides to call check_order_status tool
   ├── Tools node: gets order data from the "database"
   ├── Assistant node: LLM reads the tool results
   └── LLM generates the final response
        │
        ▼
7. Backend returns {response: "Your order is shipped...", thread_id: "thread-1707..."}
        │
        ▼
8. chat.js receives the response
   ├── Removes "Thinking..."
   ├── Adds the agent's message to the screen
   └── Auto-scrolls to the bottom
        │
        ▼
9. Customer sees the agent's response!
```

## Recap: What We Built

We created a **Next.js frontend** — a chat interface for the BigCompany Inc. customer service agent.

| File | Role |
|------|------|
| `.env` | Stores the backend URL (`http://localhost:8000`) |
| `package.json` | Lists dependencies (next, react, react-dom) |
| `next.config.mjs` | Next.js configuration |
| `jsconfig.json` | JavaScript path aliases |
| `pages/_app.js` | Loads global styles |
| `pages/_document.js` | HTML document structure |
| `pages/index.js` | Main page — assembles Layout + Chat |
| `components/layout.js` | Layout with title |
| `components/chat.js` | **The chat component** — messages, input, API calls |
| `styles/globals.css` | Base page styles |
| `styles/layout.module.css` | Layout styles |
| `styles/chat.module.css` | Chat component styles |

### The Full-Stack Architecture

```
┌─────────────────────┐       HTTP        ┌─────────────────────┐       invoke      ┌──────────────────┐
│    Frontend         │  ─────────────►   │    Backend          │  ─────────────►   │  LangGraph Agent │
│    (Next.js)        │  POST /chat       │    (FastAPI)        │                   │  (agent.py)      │
│    localhost:3000   │  ◄─────────────   │    localhost:8000   │  ◄─────────────   │                  │
│                     │  JSON response    │                     │  agent response   │  Tools:          │
│  components/chat.js │                   │  routers/chat.py    │                   │  - check_order   │
│                     │                   │  agent.py           │                   │  - return_policy │
│                     │                   │  schemas.py         │                   │  - process_return│
└─────────────────────┘                   └─────────────────────┘                   └──────────────────┘
```

## Key Takeaways

1. **The frontend is just a chat interface.** It sends messages to the backend and displays responses. No AI logic lives here.

2. **The `fetch()` call is the bridge.** The single `POST /chat` request connects the frontend to the entire LangGraph agent.

3. **Same patterns as the To Do app.** We used `useState` for state, `fetch()` for API calls, CSS Modules for styling, and `pages/index.js` for the main page. If you understand the To Do app, you understand this.

4. **Thread IDs keep conversations separate.** Each page load generates a new `thread_id`. Refreshing the page starts a new conversation.

5. **Error handling is important.** We handle two error cases: server errors (non-200 response) and connection errors (backend not running). This gives the user helpful feedback instead of a blank screen.

## Quick Reference: Running the Full-Stack App

You need **two terminals** running at the same time:

**Terminal 1 — Backend:**
```bash
cd backend
uvicorn main:app --reload
# Runs on http://localhost:8000
```

**Terminal 2 — Frontend:**
```bash
cd frontend
npm run dev
# Runs on http://localhost:3000
```

Then open **http://localhost:3000** in your browser and start chatting!